# Laya PoC - Phase 2: E1 first real run and the gate (Colab T4 from VS Code)

E1 (design doc §6.2, §5.14): `laya`, seed 11, the full train split for 4 epochs on the free T4 (13376 micro-steps, 3344 optimiser steps), resumable, with one forced crash and resume; then the temperature fit and the evaluation on val, a CPU benchmark, the B1/B3 baselines, zero-shot `laya` and the gate. Runbook: `docs/phase2_e1_runbook.md`.

**How to connect** (as for the smoke test): **Select Kernel → Colab → New Colab Server → GPU → T4**, then the Python 3
kernel. Do **not** use *Auto Connect* (it gives a CPU). Keep VS Code open and the laptop awake: a running cell keeps
the server alive. The Hugging Face token is asked in a masked input box (Step 3) and never stored in this notebook.

**After a kernel restart or a Colab disconnect:** re-run **Step 1**, then continue. Finished steps are skipped (their
outputs are on disk; `REDO = True` in Step 1 redoes them). During training: re-run Step 1, then **Step 9**, which resumes
E1 from its latest checkpoint (one every 2000 micro-steps, about 14 min apart). Step 8 skips
itself once the forced crash has happened, so **Run All** is safe. If the server itself was removed, `/content` is gone:
start again from Step 1.

**Expected runtime:** about 2.5-3.5 h: data 10-20 min, baselines 5-15, zero-shot ~2, E1 about 2-2.3 h (including
13 evals and about 6 checkpoints of ~72 s each), temperature fit ~2, evaluation ~2, CPU benchmark
10-20. Colab's free tier allows at most 12 h per session.

**Gate** (`config.yaml` `gate`, design doc §6.2; points are macro-F1 x 100), evaluated in Step 13:
- end-to-end: train → temperature fit → val evaluation → CPU benchmark all complete; at least one resume (forced here)
- numerics: no NaN/inf loss, no non-finite gradient applied, GradScaler scale ≥ 1.0 (skipped fp16 steps
  are expected and only reported)
- beats trivial: E1 val macro-F1 ≥ zero-shot `laya` + 10 points and ≥ majority + 20 points
- near TF-IDF+LR: E1 val macro-F1 ≥ TF-IDF+LR − 5 points; more than that below means
  **stop and debug** before any spend (the report lists the checklist)

The headline is 10-class macro-F1, or 9-class when the ID pool has fewer than 300
labelled Event places (`data_report.json` `headline_classes`).

Code bundle: 40 files, sha256 `8a20f4730b04ae4e290d82d5409101572c7781b074712eef427ed20477e6465e` (verified when Step 1 unpacks it).


### Step 1 - Setup
Non-interactive environment, work dir, code bundle, helpers. Seconds. **Re-run this cell first after any kernel restart or Colab disconnect.**

In [ ]:
# Step 1 - Setup
import getpass, json, os, shutil, signal, subprocess, sys, time, zipfile
from pathlib import Path

os.environ.update({
    "COLAB_DISABLE_STDIN_FOR_SHELL_MAGICS": "1",  # a command that prompts gets EOF instead of hanging
    "GIT_TERMINAL_PROMPT": "0", "PIP_NO_INPUT": "1", "PYTHONUNBUFFERED": "1", "PYTHONIOENCODING": "utf-8",
    "HF_HUB_DISABLE_PROGRESS_BARS": "1", "TOKENIZERS_PARALLELISM": "false",
    "LAYA_CUDA_AMP": "fp16",  # fp16 autocast on every card, also if the extension assigns an L4/A100
})
WORK = Path("/content/laya_poc")
DATA, RUNS = WORK / "data", WORK / "runs" / "e1"
LOGS = RUNS / "logs"
for _d in (DATA, LOGS):
    _d.mkdir(parents=True, exist_ok=True)
os.environ["LAYA_POC_ROOT"] = str(WORK)  # the CLIs read WORK/config.yaml
REDO = False  # True: recompute steps whose outputs already exist
DRILL_MARK = Path.home() / ".laya_poc_kill_drill.json"  # outside WORK: Step 13d can tell a wiped /content


def write_json(path, obj) -> None:
    """Write JSON atomically, so a kill mid-write never leaves half a file."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding="utf-8")
    os.replace(tmp, path)

def run_logged(cmd, log_path, expect_returncode=0, hint=None) -> int:
    """Run cmd with stdin closed (nothing can prompt); stream its output here and into log_path.

    VS Code truncates long cell output, so the log file is the full record. expect_returncode is an
    int, a tuple of ints, or None (accept any). Raises on an unexpected code; returns the code.
    """
    cmd, log_path = [str(c) for c in cmd], Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with open(log_path, "w", encoding="utf-8", buffering=1) as log:
        log.write("$ " + " ".join(cmd) + "\n")
        with subprocess.Popen(cmd, stdin=subprocess.DEVNULL, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, encoding="utf-8", errors="replace", bufsize=1, env=os.environ.copy()) as proc:
            try:
                for line in proc.stdout:
                    log.write(line)
                    print(line, end="", flush=True)
            except BaseException:  # interrupted cell: never leave an orphan holding the GPU
                proc.kill()
                raise
    rc = proc.returncode
    expected = (expect_returncode,) if isinstance(expect_returncode, int) else expect_returncode
    if expected is not None and rc not in expected:
        name = cmd[cmd.index("-m") + 1] if "-m" in cmd else Path(cmd[0]).name
        msg = f"{name} exited with {rc} (expected {' or '.join(map(str, expected))}); full log: {log_path}"
        raise RuntimeError(msg + (f"\nhint: {hint}" if hint else ""))
    return rc

def sh(cmd, check=True) -> str:
    """Run a short command with stdin closed; print and return its combined output."""
    res = subprocess.run(cmd, shell=isinstance(cmd, str), stdin=subprocess.DEVNULL, capture_output=True,
                         text=True, errors="replace")
    out = (res.stdout + res.stderr).strip()
    if out:
        print(out)
    if check and res.returncode:
        raise RuntimeError(f"command failed with exit code {res.returncode}: {cmd}")
    return out

def already_done(path) -> bool:
    """True (and says so) when a step's output exists and REDO is False, so re-runs skip finished steps."""
    if Path(path).exists() and not globals().get("REDO", False):
        print(f"skip: {path} exists (set REDO = True in Step 1 to recompute)")
        return True
    return False

def fresh_dir(path) -> Path:
    """Empty a run directory so a re-run starts from scratch instead of resuming a stale checkpoint."""
    path = Path(path)
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True)
    return path

def show_json(path, keys=None, width=150):
    """Print one line per top-level key (only `keys` when any of them exist), without bulky per-row or
    per-class detail; return the whole object."""
    def brief(v):
        if isinstance(v, dict):
            return {k: brief(x) for k, x in v.items() if k not in ("per_row", "per_class", "cm", "confusion")}
        return v
    obj = json.loads(Path(path).read_text(encoding="utf-8"))
    shown = brief(obj) if isinstance(obj, dict) else {"value": obj}
    if keys and any(k in shown for k in keys):
        shown = {k: shown[k] for k in keys if k in shown}
    for key, val in shown.items():
        text = json.dumps(val, ensure_ascii=False, separators=(",", ":"))
        print(f"  {key}: {text if len(text) <= width else text[:width - 3] + '...'}")
    return obj

def show_markdown(path) -> None:
    """Render a markdown file in the notebook (plain text outside IPython, e.g. in the local dry run)."""
    text = Path(path).read_text(encoding="utf-8")
    try:
        from IPython.display import Markdown, display
    except ImportError:
        print(text)
        return
    display(Markdown(text))

def restore_hf_token() -> bool:
    """Put a previously saved HF token into os.environ for the CLIs. Never prints it."""
    if os.environ.get("HF_TOKEN"):
        return True
    try:
        from huggingface_hub import constants  # importing constants does not query the Colab vault
        token_file = Path(constants.HF_TOKEN_PATH)
    except ImportError:
        token_file = Path.home() / ".cache" / "huggingface" / "token"
    token = token_file.read_text(encoding="utf-8").strip() if token_file.exists() else ""
    if token:
        os.environ["HF_TOKEN"] = token
    return bool(token)

def resume_blocker(runs, kill_at=None):
    """Why Step 10 must not run now, or None. A resume is only valid straight after the crash run: the
    resumed log must end at the injected crash (at micro-step kill_at, when given). A -9 without it is some
    other kill, e.g. the host OOM killer. A second resume (interrupted Step 10, or REDO) would restart from
    a later checkpoint and the report, which grades the last 'resumed' event, would silently FAIL."""
    runs = Path(runs)
    exit_file, log_file = runs / "crash_exit.json", runs / "resumed" / "log.jsonl"
    if not exit_file.exists():
        return "no crash_exit.json: run Step 9 (crash run) first"
    rc = json.loads(exit_file.read_text(encoding="utf-8")).get("returncode")
    if rc not in (-9, 137):
        return f"the crash run exited with {rc}, not -9/137 (hard kill)"
    events = []
    for line in log_file.read_text(encoding="utf-8").splitlines() if log_file.exists() else []:
        try:
            event = json.loads(line)
        except ValueError:  # a SIGKILL can leave half a line
            continue
        if isinstance(event, dict):
            events.append(event)
    names = [e.get("event") for e in events]
    if "crash_injected" not in names:
        return "resumed/log.jsonl has no crash_injected event"
    if names[-1] != "crash_injected":
        return "a resume already ran (or was interrupted) after the crash"
    step = events[-1].get("micro_step")
    if kill_at is not None and step != kill_at:
        return f"the crash was injected at micro-step {step}, not at the planned {kill_at}"
    return None

def log_tail(path, n=8) -> str:
    """The last n lines of a log file, for error messages ('' when it does not exist)."""
    path = Path(path)
    if not path.exists():
        return ""
    return "\n".join(path.read_text(encoding="utf-8", errors="replace").splitlines()[-n:])

def gpu_pids() -> list[int]:
    """PIDs of processes holding GPU memory according to nvidia-smi; [] without an NVIDIA driver."""
    if not shutil.which("nvidia-smi"):
        return []
    out = subprocess.run(["nvidia-smi", "--query-compute-apps=pid", "--format=csv,noheader"], text=True,
                         capture_output=True, stdin=subprocess.DEVNULL).stdout
    return [int(x) for x in out.split() if x.isdigit()]

def pid_alive(pid) -> bool:
    """True while a process with this PID exists. Linux /proc only (the drill runs on Colab); False elsewhere,
    so nothing ever signals a PID just to probe it."""
    return Path(f"/proc/{int(pid)}").exists()

def proc_cmdline(pid) -> str:
    """The command line of a process in this PID namespace ('' when it is gone or not visible)."""
    try:
        return Path(f"/proc/{int(pid)}/cmdline").read_bytes().replace(b"\0", b" ").decode("utf-8", "replace")
    except OSError:
        return ""

def drill_orphans(gpu_pids_left, child_pid, marker="LAYA_POC_KILL_DRILL"):
    """(ours, others): ours are the kill drill's own GPU holders, recognised by the marker in their command
    line, and are safe to kill; never this kernel. nvidia-smi may report host-namespace PIDs, so any other
    PID is only reported: the same number can be an unrelated process in this container."""
    me = os.getpid()
    candidates = {int(p) for p in gpu_pids_left} | {int(child_pid)}
    ours = sorted(p for p in candidates if p != me and marker in proc_cmdline(p))
    others = sorted(p for p in {int(p) for p in gpu_pids_left} if p != me and p not in ours)
    return ours, others

def read_events(path) -> list:
    """The JSON events of a run log; unparsable lines (a SIGKILL can leave half a line) are skipped."""
    path, events = Path(path), []
    for line in path.read_text(encoding="utf-8").splitlines() if path.exists() else []:
        try:
            event = json.loads(line)
        except ValueError:
            continue
        if isinstance(event, dict):
            events.append(event)
    return events

def crash_problem(exit_file, log_file, kill_at=None):
    """Why E1's forced crash has not happened as planned, or None when it has. Unlike the smoke drill, any
    number of resumes may follow it (after a Colab disconnect Step 9 simply resumes again), so the log only
    has to contain the injected crash, at micro-step kill_at when given."""
    exit_file = Path(exit_file)
    if not exit_file.exists():
        return "no crash_exit.json: the crash run (Step 8) has not finished"
    rc = json.loads(exit_file.read_text(encoding="utf-8")).get("returncode")
    if rc not in (-9, 137):
        return f"the crash run exited with {rc}, not -9/137 (hard kill)"
    crashes = [e for e in read_events(log_file) if e.get("event") == "crash_injected"]
    if not crashes:
        return f"{Path(log_file).name} has no crash_injected event"
    step = crashes[0].get("micro_step")
    if kill_at is not None and step != kill_at:
        return f"the crash was injected at micro-step {step}, not at the planned {kill_at}"
    return None

def next_log(logs, stem) -> Path:
    """logs/<stem>.log, or <stem>_2.log, <stem>_3.log, ... when earlier attempts exist: a resume after a
    disconnect keeps the logs of the attempts before it."""
    logs, n = Path(logs), 1
    while (logs / (f"{stem}.log" if n == 1 else f"{stem}_{n}.log")).exists():
        n += 1
    return logs / (f"{stem}.log" if n == 1 else f"{stem}_{n}.log")

def trainer_pids(module="laya_poc.train_single") -> list:
    """PIDs of running trainers other than this kernel (Linux /proc; [] elsewhere). A trainer started before a
    kernel restart can outlive the kernel for a while: a second one must never start on the same run dir."""
    proc = Path("/proc")
    if not proc.is_dir():
        return []
    pids = [int(p.name) for p in proc.iterdir() if p.name.isdigit() and int(p.name) != os.getpid()]
    return sorted(pid for pid in pids if module in proc_cmdline(pid))

def fnum(value, nd=4) -> str:
    """A metric for a one-line summary: rounded when numeric, else as is (None when absent)."""
    return f"{value:.{nd}f}" if isinstance(value, float) else str(value)

def dig(obj, *keys):
    """obj[k1][k2]... or None when any level is missing (to print optional report fields)."""
    for key in keys:
        if not isinstance(obj, dict) or key not in obj:
            return None
        obj = obj[key]
    return obj


SENTINEL = WORK / "sentinel.json"
if SENTINEL.exists():
    print("sentinel from", json.loads(SENTINEL.read_text(encoding="utf-8"))["created"], "found: /content survived")
else:
    write_json(SENTINEL, {"created": time.strftime("%Y-%m-%d %H:%M:%S"), "pid": os.getpid()})
    print("sentinel created: first Step 1 on this server (if Step 1 ran here before, /content was wiped)")
print("HF token: restored from the token file" if restore_hf_token() else "HF token: not set yet (Step 3)")

import base64, hashlib, io, pathlib, zipfile
def _unpack(b64, sha256, dest):
    raw = base64.b64decode(b64)
    assert hashlib.sha256(raw).hexdigest() == sha256, "bundle checksum mismatch"
    dest = pathlib.Path(dest); dest.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(raw)) as zf:
        for n in zf.namelist():
            assert dest.resolve() in (dest / n).resolve().parents, f"unsafe path {n}"
        zf.extractall(dest)
        return zf.namelist()
BUNDLE_SHA256 = "8a20f4730b04ae4e290d82d5409101572c7781b074712eef427ed20477e6465e"
BUNDLE_B64 = "..."
print(f"unpacked {len(_unpack(BUNDLE_B64, BUNDLE_SHA256, WORK))} files into {WORK}; bundle sha256 {BUNDLE_SHA256[:16]}")


### Step 2 - Install
Plain `laya` at the pinned commit, DuckDB pinned to 1.5.5 (`data.duckdb_version`: the frozen splits depend on its hash()), this project. Fails if pip changed Colab's torch, transformers, protobuf or numpy. 1-3 min.

In [ ]:
# Step 2 - Install
import importlib, importlib.metadata as md

LAYA_COMMIT = "4066d5d5fbf08b66c6757ddeedbd797bd7655bc0"
PIP = [sys.executable, "-m", "pip", "install", "-q", "--no-input", "--progress-bar", "off"]


def dist_version(name):
    try:
        return md.version(name)
    except md.PackageNotFoundError:
        return None


def direct_url(name):  # PEP 610 metadata: what pip actually installed, and from where
    try:
        return json.loads(md.distribution(name).read_text("direct_url.json") or "{}")
    except md.PackageNotFoundError:
        return {}


before = {p: dist_version(p) for p in ("torch", "transformers", "protobuf", "numpy")}  # Colab's own builds
if direct_url("laya").get("vcs_info", {}).get("commit_id") != LAYA_COMMIT:  # no [onnx] extra: it pulls protobuf 6+
    run_logged(PIP + [f"laya @ git+https://github.com/NandhaKishorM/laya.git@{LAYA_COMMIT}"], LOGS / "02_install_laya.log")
DUCKDB = "1.5.5"  # config data.duckdb_version: hash()-based sampling must match the frozen build exactly
if dist_version("duckdb") != DUCKDB:
    run_logged(PIP + [f"duckdb=={DUCKDB}"], LOGS / "02_install_duckdb.log")
if not direct_url("laya-poc").get("dir_info", {}).get("editable"):
    run_logged(PIP + ["--no-deps", "-e", str(WORK)], LOGS / "02_install_project.log")
importlib.invalidate_caches()
changed = {p: f"{before[p]} -> {dist_version(p)}" for p in before if dist_version(p) != before[p]}
if changed:
    raise RuntimeError(f"pip changed {changed}; run Colab: Remove Server and start again")
got = direct_url("laya").get("vcs_info", {}).get("commit_id")
if got != LAYA_COMMIT:
    raise RuntimeError(f"installed laya commit {got} != pinned {LAYA_COMMIT}")
if dist_version("duckdb") != DUCKDB:
    raise RuntimeError(f"duckdb {dist_version('duckdb')} != pinned {DUCKDB}; see {LOGS / '02_install_duckdb.log'}")
print({p: dist_version(p) for p in ("torch", "transformers", "huggingface_hub", "duckdb", "laya", "laya-poc")},
      "laya commit", got[:12])


### Step 3 - Hugging Face token
Masked input box at the top of VS Code; never printed. Unattended: `python tools/build_e1_notebook.py --with-token` and open the gitignored `notebooks/e1_first_run.local.ipynb`.

In [ ]:
# Step 3 - Hugging Face token
import re
from huggingface_hub import login

PRESET_TOKEN = ""  # filled only in the gitignored e1_first_run.local.ipynb (build_e1_notebook.py --with-token)
ASK_AGAIN = False  # True: ignore the saved token (e.g. one without access to the gated dataset) and ask for a new one
NEW_TOKEN = "set ASK_AGAIN = True at the top of this cell and re-run it to enter a new token"
restore_hf_token()
token = os.environ.pop("HF_TOKEN", "").strip()
if PRESET_TOKEN and not ASK_AGAIN:  # unattended run: no prompt
    token = PRESET_TOKEN.strip()
del PRESET_TOKEN
if ASK_AGAIN or not token:  # VS Code shows a masked input box at the top of the window
    token = getpass.getpass("Hugging Face token (read access to foursquare/fsq-os-places; hidden): ").strip()
if not token:
    raise RuntimeError("no token entered (Esc/cancel sends an empty string); re-run this cell and paste it")
if not re.fullmatch(r"hf_[A-Za-z0-9]{20,}", token):  # never echo it: a line break would leak the rest in errors
    del token
    raise RuntimeError("that does not look like a Hugging Face token (hidden characters or line breaks?): "
                       + NEW_TOKEN)
try:
    login(token=token)  # validates with whoami and saves the token file, so a restart does not re-prompt
except Exception as exc:
    status = getattr(getattr(exc, "response", None), "status_code", None)
    if status in (401, 403):
        raise RuntimeError(f"the Hub rejected the token (HTTP {status}): {NEW_TOKEN}") from None
    text = str(exc).replace(token, "***")[:200]  # the exact token first, then anything token-like
    detail = re.sub(r"hf_\w+", "hf_***", type(exc).__name__ + (f", HTTP {status}" if status else "") + f": {text}")
    raise RuntimeError(f"could not reach the Hub to check the token ({detail}); check the network and re-run this "
                       f"cell, or {NEW_TOKEN}") from None
os.environ["HF_TOKEN"] = token  # inherited by the CLIs (DuckDB reads the gated dataset with it)
del token
print("HF token OK (hidden)")


### Step 4 - Environment check
GPU, `sm_75`, driver, VRAM, disk, versions, Laya commit -> `runs/e1/env.json`. ~30 s.

In [ ]:
# Step 4 - Environment check
if shutil.which("nvidia-smi"):
    sh(["nvidia-smi"], check=False)
env_json = RUNS / "env.json"
env_json.unlink(missing_ok=True)  # a stale file must not explain this run's failure
cmd = [sys.executable, "-m", "laya_poc.env_check", "--out", str(RUNS / "env.json"), "--require-gpu",
       "--expect-card", "T4"]
if run_logged(cmd, LOGS / "04_env.log", expect_returncode=None):  # each error in env.json names its fix
    errors = json.loads(env_json.read_text(encoding="utf-8")).get("errors") if env_json.exists() else None
    detail = "; ".join(errors) if errors else f"no env.json written; see {LOGS / '04_env.log'}"
    raise RuntimeError(f"environment check failed: {detail} (runbook: When something fails)")
env = show_json(env_json, ["gpu_name", "capability", "has_sm75", "driver", "vram_total_gb", "disk_free_gb", "torch", "transformers", "laya_commit", "card", "warnings"])


### Step 5 - Full data
FULL build over `hf://` (two passes, all 15 countries), trap candidates held out of every split, JSONL, leakage asserts, then `--verify-frozen`: the JSONL fingerprint must equal the frozen manifest (`data.frozen_manifest`), else the step fails. 10-20 min.

In [ ]:
# Step 5 - Full data
FROZEN = 'docs/results/data_fingerprint_2026-09-15.json'  # config data.frozen_manifest: --verify-frozen compares the build with it
if not already_done(DATA / "build_ok.json"):
    (DATA / "build_ok.json").unlink(missing_ok=True)
    cmd = [sys.executable, "-m", "laya_poc.build_data", "--out", str(DATA), "--verify-frozen"]
    run_logged(cmd, LOGS / "05_data.log",
               hint="401/403: accept the terms of the gated dataset foursquare/fsq-os-places on the Hub; for a different token set ASK_AGAIN = True in Step 3 and re-run it; then re-run this step; if the fingerprint differs from the frozen manifest, this build is not the frozen E1 data: compare the versions in data_report.json with the manifest and do not train on it (runbook)")
    write_json(DATA / "build_ok.json", {"frozen_manifest": FROZEN, "time": time.strftime("%Y-%m-%d %H:%M")})
data_report = show_json(DATA / "data_report.json", ["split_sizes", "headline_rule", "traps", "fingerprint_sha256", "frozen_check", "warnings"])


### Step 6 - Baselines B1 + B3
Majority / prior and char TF-IDF + logistic regression (C tuned and T fitted on val), 10- and 7-class, on the CPU. 5-15 min.

In [ ]:
# Step 6 - Baselines B1 + B3
if not already_done(RUNS / "baselines.json"):
    cmd = [sys.executable, "-m", "laya_poc.baselines", "--data-dir", str(DATA),
           "--out", str(RUNS / "baselines.json")]
    run_logged(cmd, LOGS / "06_baselines.log")
baselines = json.loads((RUNS / "baselines.json").read_text(encoding="utf-8"))
for scheme in ("c10", "c7"):
    arms = dig(baselines, "schemes", scheme) or {}
    print(f"{scheme} val macro-F1: B1 majority {fnum(dig(arms, 'b1_majority', 'val', 'macro_f1'))}, B3 TF-IDF+LR "
          f"{fnum(dig(arms, 'b3_tfidf_lr', 'val', 'post', 'macro_f1'))} (C {dig(arms, 'b3_tfidf_lr', 'C')}, "
          f"T {fnum(dig(arms, 'b3_tfidf_lr', 'T'), 3)})")


### Step 7 - Zero-shot laya on val
The shipped checkpoint on the full val split (the gate's 'beats zero-shot' reference). ~2 min. `ZERO_SHOT_ML = True` adds laya-multilingual.

In [ ]:
# Step 7 - Zero-shot laya on val
ZERO_SHOT_ML = False  # True: also zero-shot laya-multilingual (B2; ~1.2 GB download); the gate shows it
if not already_done(RUNS / "eval_zeroshot_val.json"):
    cmd = [sys.executable, "-m", "laya_poc.evaluate", "--ckpt", "hub", "--model", "laya",
           "--rows", str(DATA / "val.jsonl"), "--out", str(RUNS / "eval_zeroshot_val.json"),
           "--device", "cuda"]
    run_logged(cmd, LOGS / "07_zeroshot.log")
if ZERO_SHOT_ML:
    if not already_done(RUNS / "eval_zeroshot_ml_val.json"):
        cmd = [sys.executable, "-m", "laya_poc.evaluate", "--ckpt", "hub", "--model", "laya_ml",
               "--rows", str(DATA / "val.jsonl"), "--out", str(RUNS / "eval_zeroshot_ml_val.json"),
               "--device", "cuda"]
        run_logged(cmd, LOGS / "07_zeroshot_ml.log")
for name in ("eval_zeroshot_val.json", "eval_zeroshot_ml_val.json"):
    if (RUNS / name).exists():
        print(name)
        zs = show_json(RUNS / name, ["model", "n", "n_unlabelled", "n_abstained_no_evidence", "temperature", "pre", "post", "cpu_fallback", "seconds"])


### Step 8 - E1 crash run
E1 from scratch: `laya`, seed 11, 4 epochs x 3344 micro-steps (MB 8 x ACC 4, `--card T4`), a checkpoint every 2000 micro-steps, hard-killed right after micro-step 2002 (exit -9/137, `crash_exit.json`). Skipped once the crash has happened, so Run All never restarts E1. 15-20 min.

In [ ]:
# Step 8 - E1 crash run
KILL_AT = 2002  # config e1.crash_at_micro_step: the resume restarts from the micro-step-2000 checkpoint
TRAIN, OOM_HINT = RUNS / "train", "CUDA OOM: add '--micro-batch', '4', '--effective-batch', '32' to cmd in Steps 8 and 9 alike (the resume refuses other settings), delete runs/e1/crash_exit.json so Step 8 starts E1 afresh, then run Steps 8 and 9 (runbook: When something fails)"
if already_done(TRAIN / "summary.json"):
    pass
elif not REDO and crash_problem(RUNS / "crash_exit.json", TRAIN / "log.jsonl", KILL_AT) is None:
    print(f"skip: E1 was already hard-killed at micro-step {KILL_AT} as planned; run Step 9 (it resumes E1)")
elif trainer_pids():
    raise RuntimeError(f"a trainer is still running (pid {trainer_pids()}, from before a kernel restart?): wait "
                       "for it to stop, then re-run this step")
else:
    fresh_dir(TRAIN)  # the crash run always starts E1 from scratch
    (RUNS / "crash_exit.json").unlink(missing_ok=True)
    cmd = [sys.executable, "-m", "laya_poc.train_single", "--run-dir", str(RUNS / "train"),
           "--data-dir", str(DATA), "--model", "laya", "--seed", "11", "--epochs", "4", "--device", "cuda",
           "--card", "T4", "--ckpt-every-micro-steps", "2000", "--ckpt-every-min", "0", "--initial-eval",
           "--final-eval", "--save-final", "--save-best", "--crash-at-micro-step", "2002"]
    rc = run_logged(cmd, LOGS / "08_e1_crash.log", expect_returncode=None)
    write_json(RUNS / "crash_exit.json", {"returncode": rc})
    tail = "\n" + log_tail(LOGS / "08_e1_crash.log")
    if rc not in (-9, 137):
        raise RuntimeError(f"the E1 crash run exited with {rc}, not -9/137 (hard kill); if CUDA OOM: {OOM_HINT}" + tail)
    problem = crash_problem(RUNS / "crash_exit.json", TRAIN / "log.jsonl", KILL_AT)
    if problem:  # e.g. the host OOM killer, not the injected crash
        raise RuntimeError(f"E1 was killed ({rc}) but not as planned: {problem}. A kill without crash_injected is "
                           "usually the host OOM killer: check RAM (free -h) and the kernel log (dmesg | tail), "
                           "then re-run this step." + tail)
    print("E1 was hard-killed as planned at micro-step", KILL_AT, "- exit code", rc)


### Step 9 - E1 resume (to the end)
Same settings without the crash flag: resumes from micro-step 2000 and trains to the end or an early stop, evaluating val every 250 opt steps (13 evals), exporting `best/` and `final/`. **After a disconnect: re-run Step 1, then this step**; it resumes from the latest checkpoint. About 1.5-2 h.

In [ ]:
# Step 9 - E1 resume (to the end)
KILL_AT, TRAIN = 2002, RUNS / "train"
if not already_done(TRAIN / "summary.json"):
    problem = crash_problem(RUNS / "crash_exit.json", TRAIN / "log.jsonl", KILL_AT)
    if problem:
        raise RuntimeError(f"E1 cannot resume yet ({problem}): run Step 8 first")
    if trainer_pids():  # e.g. the run from before a kernel restart: two trainers must never share a run dir
        raise RuntimeError(f"an E1 trainer is still running (pid {trainer_pids()}): wait for it to stop, or stop "
                           "it with os.kill(pid, 9) (checkpoints are written atomically), then re-run this step")
    log = next_log(LOGS, "09_e1_resume")  # one log per attempt: a resume after a disconnect keeps the earlier ones
    cmd = [sys.executable, "-m", "laya_poc.train_single", "--run-dir", str(RUNS / "train"),
           "--data-dir", str(DATA), "--model", "laya", "--seed", "11", "--epochs", "4", "--device", "cuda",
           "--card", "T4", "--ckpt-every-micro-steps", "2000", "--ckpt-every-min", "0", "--initial-eval",
           "--final-eval", "--save-final", "--save-best"]
    run_logged(cmd, log, hint="CUDA OOM: add '--micro-batch', '4', '--effective-batch', '32' to cmd in Steps 8 and 9 alike (the resume refuses other settings), delete runs/e1/crash_exit.json so Step 8 starts E1 afresh, then run Steps 8 and 9 (runbook: When something fails)")
    if "resumed from" not in log.read_text(encoding="utf-8"):
        raise RuntimeError("no 'resumed from ...' line: E1 restarted from scratch instead of resuming")
e1 = show_json(TRAIN / "summary.json", ["micro_steps", "opt_steps", "stop_reason", "resumed_from", "evals", "best_opt_step", "best_eval", "peak_vram_reserved_gb", "sec_per_micro_mean", "nonfinite", "opt_steps_skipped", "nonfinite_grad_applied", "min_scale"])


### Step 10 - Temperature fit
T fitted on raw val logits of `best/`, written into it, reloaded and checked against `best/train_eval.json`. ~2 min.

In [ ]:
# Step 10 - Temperature fit
if not already_done(RUNS / "export_check.json"):
    cmd = [sys.executable, "-m", "laya_poc.export_check", "--ckpt", str(RUNS / "train" / "best"),
           "--rows", str(DATA / "val.jsonl"), "--out", str(RUNS / "export_check.json"), "--device", "cuda",
           "--train-summary", str(RUNS / "train" / "best" / "train_eval.json")]
    run_logged(cmd, LOGS / "10_export_check.log")
export = show_json(RUNS / "export_check.json", ["T", "clamped", "roundtrip_max_dp", "passed", "cpu_fallback", "train_eval"])
if not export.get("passed"):
    print("WARNING: export_check did not pass (see train_eval, budgets, roundtrip_max_dp above); the gate will FAIL end-to-end")


### Step 11 - Evaluate E1 on val
`best/` with and without its temperature: macro-F1 (10- and 9-class), ECE, selective accuracy; per-row predictions. ~2 min.

In [ ]:
# Step 11 - Evaluate E1 on val
if not already_done(RUNS / "eval_e1_val.json"):
    cmd = [sys.executable, "-m", "laya_poc.evaluate", "--ckpt", str(RUNS / "train" / "best"),
           "--model", "laya", "--rows", str(DATA / "val.jsonl"), "--out", str(RUNS / "eval_e1_val.json"),
           "--preds", str(RUNS / "preds" / "e1_val.jsonl"), "--device", "cuda"]
    run_logged(cmd, LOGS / "11_evaluate.log")
e1_val = show_json(RUNS / "eval_e1_val.json", ["model", "n", "n_unlabelled", "n_abstained_no_evidence", "temperature", "pre", "post", "cpu_fallback", "seconds"])


### Step 12 - CPU benchmark
200 test_id records on this host's CPU, threads [1, 2] (a fresh process each): cold start, p50/p95 at batch 1, batched rec/s. Information only (the decision rule's CPU budget is Phase 5, on 4 vCPUs). 10-20 min.

In [ ]:
# Step 12 - CPU benchmark
if not already_done(RUNS / "bench_ok.json"):
    (RUNS / "bench_ok.json").unlink(missing_ok=True)
    cmd = [sys.executable, "-m", "laya_poc.bench_cpu", "--ckpt", str(RUNS / "train" / "best"),
           "--model", "laya", "--rows", str(DATA / "test_id.jsonl"), "--n", "200", "--threads", "1", "2",
           "--warmup", "20", "--batch-size", "32", "--out", str(RUNS / "bench_cpu.json")]
    run_logged(cmd, LOGS / "12_bench_cpu.log")
    write_json(RUNS / "bench_ok.json", {"time": time.strftime("%Y-%m-%d %H:%M")})
bench = show_json(RUNS / "bench_cpu.json")


### Step 13 - Gate
Design doc §6.2 gate table -> `runs/e1/gate_report.md` (+ `.json`).

In [ ]:
# Step 13 - Gate
cmd = [sys.executable, "-m", "laya_poc.gate", "--run-root", str(RUNS), "--data-dir", str(DATA),
       "--out", str(RUNS / "gate_report.md")]
run_logged(cmd, LOGS / "13_gate.log")
show_markdown(RUNS / "gate_report.md")


### Step 14a - Archive (optional)
---
Zip logs, JSON, the gate report and the FSQ NOTICE (no checkpoints, no FSQ rows) for **Download...**; optional Drive copy of the zip and of `best/`.

In [ ]:
# Step 14a - Archive (optional)
MAKE_ARCHIVE = True         # logs, JSON, markdown, the gate report and the FSQ NOTICE: no checkpoints, no FSQ rows
COPY_TO_DRIVE = False       # interactive Google sign-in (about 2 min); fails for some Workspace accounts
COPY_BEST_TO_DRIVE = False  # also copy the E1 best/ checkpoint (about 0.9 GB fp16, with its NOTICE files)
ARCHIVE = WORK / "e1_artifacts.zip"
if MAKE_ARCHIVE:
    small = {".json", ".jsonl", ".md", ".log", ".yaml"}
    keep = [p for p in sorted(RUNS.rglob("*")) if p.is_file() and p.suffix in small
            and not {"ckpt", "best", "best.prev", "final"} & set(p.relative_to(RUNS).parts)]
    keep += [p for p in (DATA / "data_report.json", DATA / "SHA256SUMS", DATA / "NOTICE_FSQ.txt") if p.exists()]
    with zipfile.ZipFile(ARCHIVE, "w", zipfile.ZIP_DEFLATED) as zf:
        for p in keep:
            zf.write(p, p.relative_to(WORK).as_posix())
    print(f"{ARCHIVE}: {len(keep)} files; Colab view > Contents > right-click > Download...")
if COPY_TO_DRIVE or COPY_BEST_TO_DRIVE:
    from google.colab import drive
    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive")
    dest = Path("/content/drive/MyDrive/laya_poc")
    dest.mkdir(parents=True, exist_ok=True)
    if COPY_TO_DRIVE and ARCHIVE.exists():
        print("copied to", shutil.copy2(ARCHIVE, dest / ARCHIVE.name))
    if COPY_BEST_TO_DRIVE:
        print("copied to", shutil.copytree(RUNS / "train" / "best", dest / "e1_best", dirs_exist_ok=True))


### Step 14b - Free disk (optional)
Delete the resumable checkpoints once E1 has finished.

In [ ]:
# Step 14b - Free disk (optional)
CLEANUP_CKPTS = False  # True: delete the resumable checkpoints (about 5 GB each); best/ and final/ are kept
if CLEANUP_CKPTS and not (RUNS / "train" / "summary.json").exists():
    print("E1 has not finished: keeping its checkpoints (Step 9 resumes from them)")
elif CLEANUP_CKPTS:
    for ckpt_dir in sorted(RUNS.glob("*/ckpt")):
        size = sum(f.stat().st_size for f in ckpt_dir.rglob("*") if f.is_file())
        shutil.rmtree(ckpt_dir)
        print(f"removed {ckpt_dir} ({size / 1e9:.1f} GB)")
else:
    print("checkpoint cleanup skipped (CLEANUP_CKPTS = False)")


### Finish
- The verdict is at the end of Step 13; details in `runs/e1/gate_report.md` and `gate_report.json`.
- Download `e1_artifacts.zip` (Step 14a): Colab view in the activity bar > Contents > right-click > **Download...**
- Pending human task (not part of the gate): trap annotation. `data/trap_candidates.csv` holds FSQ rows: two annotators
  mark `keep_a1` / `keep_a2`, the lead resolves disagreements, then `python -m laya_poc.traps merge` writes
  `data/trap.jsonl` (runbook). Download the CSV for the annotators only if the FSQ terms allow it for your team.
- Run **Colab: Remove Server** when you are done. It stops quota use and deletes `/content`, checkpoints included.
